# Module 02 — DuckDB

**Formation Big Data — ANSD / Data Innovation Lab**

Deuxième outil du bloc, et changement complet de point de vue : DuckDB n'est pas
une bibliothèque de manipulation de tableaux, c'est une **base de données
analytique qui s'exécute dans votre processus Python**. Pas de serveur, pas
d'administration, pas de configuration — une simple bibliothèque, et du SQL.

On la présente souvent comme « le SQLite de l'analyse ».

Au programme :

1. interroger un fichier CSV **sans le charger** ;
2. le profilage automatique avec `SUMMARIZE` ;
3. les trois calculs métier, en SQL ;
4. lire le plan d'exécution avec `EXPLAIN ANALYZE` ;
5. l'interopérabilité avec pandas et Polars, **sans recopie** ;
6. la campagne de mesures.

## 1. Protocole et contexte

In [ ]:
%load_ext autoreload 
%autoreload 2
import sys
from pathlib import Path
try:
    sys.path.append(str(Path(__file__).parent.parent.resolve()))
except NameError:
    sys.path.append(str(Path.cwd().parent.resolve()))
from tools.outils_mesure import (FICHIER, FICHIER_REGIONS, VOLUME_COMPARAISON,
                           afficher_protocole, contexte_machine, enregistrer,
                           memoire_mo, mesurer)

import duckdb

print("DuckDB", duckdb.__version__)
print()
contexte_machine()

In [ ]:
# Une connexion, en mémoire. La barre de progression est désactivée pour ne
# pas polluer les mesures ; réactivez-la si vous préférez la voir.
con = duckdb.connect()
con.sql("SET enable_progress_bar = false")

# Chemin du fichier, au format attendu par SQL
CHEMIN = str(FICHIER).replace("\\", "/")
CHEMIN

## 2. La ligne qui change tout

Aucun chargement, aucune déclaration de schéma, aucun serveur. On interroge le
fichier directement.

In [ ]:
con.sql(f"""
    SELECT region, COUNT(*) AS effectif
    FROM '{CHEMIN}'
    GROUP BY region
    ORDER BY effectif DESC
    LIMIT 5
""")

Prenez la mesure de ce qui vient de se passer : DuckDB a lu un fichier de
plusieurs centaines de mégaoctets, détecté seul le séparateur, les noms et les
types de colonnes, effectué l'agrégation — et il n'a jamais chargé le fichier
en mémoire. Il n'a lu que la colonne `region`.

**Question 1.** D'après ce que vous avez vu chez Polars, quel levier DuckDB
vient-il d'utiliser ici ?

*Votre réponse :* …

### Profilage automatique

`SUMMARIZE` produit en une commande ce que l'on écrit habituellement en dix
lignes : types, bornes, cardinalité, taux de valeurs manquantes, quartiles.

In [ ]:
con.sql(f"""
    SUMMARIZE
    SELECT region, age, sexe, niveau_instruction, situation_activite
    FROM '{CHEMIN}'
""")

**Question 2.** Repérez dans ce tableau deux des anomalies que vous aviez
identifiées à la main au notebook 01. Lesquelles, et sur quelles colonnes ?

*Votre réponse :* …

## 3. Charger, ou ne pas charger ?

Interroger le fichier directement est très pratique, mais chaque requête
**relit et réanalyse le CSV**. Si vous enchaînez dix requêtes, vous payez dix
fois ce coût.

L'alternative : charger une fois dans une table DuckDB, puis interroger cette
table. Mesurons la différence.

In [ ]:
# Lecture à blanc, pour que le cache du système soit dans le même état
_ = con.sql(f"SELECT COUNT(*) FROM '{CHEMIN}'").fetchone()

requete = """
    SELECT trim(region) AS region, COUNT(*) AS effectif, AVG(age) AS age_moyen
    FROM {source}
    WHERE age BETWEEN 0 AND 110
    GROUP BY 1
    ORDER BY effectif DESC
"""

print("Requête directe sur le fichier CSV :")
m_fichier = mesurer("sur CSV", lambda: con.sql(
    requete.format(source=f"'{CHEMIN}'")).fetchall())

print("\nChargement dans une table DuckDB :")
m_charge = mesurer("chargement", lambda: con.sql(
    f"CREATE OR REPLACE TABLE individus AS SELECT * FROM '{CHEMIN}'"))

print("\nMême requête, sur la table chargée :")
m_table = mesurer("sur table", lambda: con.sql(
    requete.format(source="individus")).fetchall())

**Question 3.** Combien de requêtes faut-il enchaîner pour que le
chargement préalable devienne rentable ? Faites le calcul avec vos propres
mesures.

*Votre réponse :* …

> Retenez ce raisonnement : il reviendra au bloc suivant. Le coût que l'on paie
> ici, c'est celui de l'**analyse du CSV** — un format texte qu'il faut découper
> et convertir ligne à ligne. Un format binaire et colonnaire supprimerait
> l'essentiel de ce coût.

## 4. Les trois calculs métier, en SQL

Les mêmes qu'aux notebooks précédents. Terrain familier : c'est du SQL
standard.

In [ ]:
# À COMPLÉTER — effectifs par région, libellés nettoyés,
# du plus peuplé au moins peuplé.
#
# Attention : DuckDB n'a pas de fonction de mise en capitale initiale.
# La bonne pratique est ailleurs — normalisez les libellés en une CLÉ
# (par exemple upper(trim(region))), puis récupérez le libellé propre
# depuis la table de référence regions.csv en joignant sur cette clé.
# C'est exactement ainsi que l'on redresse un fichier administratif.

con.sql("""
    ...
""")

In [ ]:
# À COMPLÉTER — âge moyen par sexe, en excluant les âges aberrants.

con.sql("""
    ...
""")

In [ ]:
# À COMPLÉTER — taux d'activité par milieu de résidence chez les 15 ans et plus.
# Définition : part des personnes « Occupé » ou « Chômeur ».
#
# Indice : AVG(CASE WHEN ... THEN 1 ELSE 0 END) donne directement une proportion,
# ou plus court : AVG((situation_activite IN (...))::INT)

con.sql("""
    ...
""")

### Une jointure supplémentaire : la densité par région

L'exercice précédent vous a fait joindre la table de référence pour récupérer
les libellés. Prolongez-le : la table `regions.csv` porte aussi les
superficies.

In [ ]:
# À COMPLÉTER — calculez le nombre d'individus par km² et par région,
# du plus dense au moins dense.

CHEMIN_REGIONS = str(FICHIER_REGIONS).replace("\\", "/")

con.sql(f"""
    ...
""")

## 5. Lire le plan d'exécution

Comme Polars, DuckDB optimise avant d'exécuter. `EXPLAIN ANALYZE` affiche le
plan **et** le temps passé à chaque étape.

In [ ]:
plan = con.sql(f"""
    EXPLAIN ANALYZE
    SELECT trim(region) AS region, COUNT(*) AS effectif
    FROM '{CHEMIN}'
    WHERE age BETWEEN 15 AND 110
    GROUP BY 1
""").fetchall()

print(plan[0][1])

**Question 4.** Dans l'arbre ci-dessus, repérez l'étape de lecture du
fichier. Combien de colonnes sont lues ? Le filtre sur l'âge apparaît-il avant
ou après la lecture ?

*Votre réponse :* …

**Question 5.** Comparez avec le plan produit par Polars au notebook précédent.
Les deux moteurs prennent-ils les mêmes décisions ?

*Votre réponse :* …

## 6. Interopérabilité : le meilleur des deux mondes

DuckDB voit les variables Python de votre session. Un DataFrame pandas ou
Polars peut donc être interrogé **directement en SQL**, sans conversion ni
recopie — les trois outils partagent le même format mémoire (Arrow).

In [ ]:
import pandas as pd
import polars as pl

# Un DataFrame pandas ordinaire
echantillon_pandas = pd.read_csv(FICHIER, nrows=100_000)

# … interrogé en SQL, sans rien déclarer
con.sql("""
    SELECT sexe, COUNT(*) AS effectif, AVG(age) AS age_moyen
    FROM echantillon_pandas
    GROUP BY sexe
""")

In [ ]:
# Et dans l'autre sens : le résultat d'une requête, rendu dans l'outil de
# votre choix
resultat_pandas = con.sql("SELECT region, COUNT(*) AS n FROM individus GROUP BY 1").df()
resultat_polars = con.sql("SELECT region, COUNT(*) AS n FROM individus GROUP BY 1").pl()

print(type(resultat_pandas), "|", type(resultat_polars))
resultat_polars.head(3)

**C'est la pratique réelle en 2026** : on ne choisit pas *un* outil, on
compose. Une requête d'agrégation en SQL avec DuckDB, une transformation
complexe en Polars, un graphique depuis pandas — sans jamais payer de
conversion.

**Question 6.** Dans votre travail à l'ANSD, quelles tâches confieriez-vous
plutôt à SQL, et lesquelles plutôt à une API de type DataFrame ?

*Votre réponse :* …

## 7. Deux fonctions utiles au quotidien

### Lire plusieurs fichiers d'un seul coup

Un motif suffit : `'donnees/*.csv'` lit tous les fichiers correspondants comme
une seule table. Très utile pour des fichiers découpés par région ou par année.

### Travailler au-delà de la mémoire disponible

DuckDB déborde automatiquement sur le disque quand la mémoire manque. On peut
lui fixer une limite explicite.

In [ ]:
# Fixer une limite mémoire : DuckDB s'y tiendra, en écrivant sur disque si besoin
con.sql("SET memory_limit = '2GB'")
con.sql("SELECT current_setting('memory_limit') AS limite_memoire")

In [ ]:
# Une base persistante, si l'on veut conserver les tables entre deux sessions
# (fichier unique, aucun serveur à administrer)
#
# con_disque = duckdb.connect("entrepot_ansd.duckdb")
# con_disque.sql("CREATE TABLE individus AS SELECT * FROM 'individus.csv'")
print("Base persistante : un simple fichier, aucune administration.")

## 8. Campagne de mesures

Cinq opérations, dans l'ordre du protocole, sur le volume de comparaison.

Pour rester comparable aux autres outils, l'opération « lecture » correspond au
**chargement dans une table DuckDB** : c'est l'équivalent d'un `read_csv`. Les
autres opérations s'exécutent ensuite sur cette table.

In [ ]:
# Fourni : préparation de la campagne
con.sql(f"""
    CREATE OR REPLACE TABLE mesure AS
    SELECT * FROM '{CHEMIN}' LIMIT {VOLUME_COMPARAISON}
""")
con.sql("""
    CREATE OR REPLACE TABLE reference AS
    SELECT id_individu, nom FROM mesure USING SAMPLE 50 PERCENT (bernoulli, 1)
""")
print(con.sql("SELECT COUNT(*) AS lignes FROM mesure").fetchone()[0], "lignes")

In [ ]:
# À COMPLÉTER — mesurez les cinq opérations, dans cet ordre exact.
# Pensez à terminer chaque requête par .fetchall() ou .df() : sans cela,
# DuckDB pourrait n'avoir rien exécuté au moment où le chronomètre s'arrête.
#
#   lecture     : CREATE OR REPLACE TABLE ... AS SELECT ... LIMIT VOLUME_COMPARAISON
#   filtre      : les 15 ans et plus
#   agregation  : âge moyen par région
#   tri         : tri par région puis âge
#   jointure    : jointure de `mesure` avec `reference` sur id_individu

mesures = []
mesures.append(mesurer("lecture",    lambda: ...))
mesures.append(mesurer("filtre",     lambda: ...))
mesures.append(mesurer("agregation", lambda: ...))
mesures.append(mesurer("tri",        lambda: ...))
mesures.append(mesurer("jointure",   lambda: ...))

enregistrer(mesures, outil="duckdb", volume="2M", lignes=VOLUME_COMPARAISON)

> ⚠️ **Un piège propre à DuckDB.** Une requête n'est réellement exécutée
> qu'au moment où l'on en demande le résultat. Si vous chronométrez
> `con.sql(...)` sans `.fetchall()`, vous mesurez la construction du plan, pas
> le calcul — et vous obtiendrez des temps absurdement bas. Vérifiez vos
> mesures : si une opération affiche moins d'un millième de seconde, c'est
> probablement ce qui s'est produit.

### Passe sur le fichier complet

In [ ]:
# À COMPLÉTER — répétez les cinq mesures sur le fichier complet (sans LIMIT).
# Un échec éventuel sera enregistré comme tel.

con.sql(f"CREATE OR REPLACE TABLE mesure AS SELECT * FROM '{CHEMIN}'")
con.sql("""
    CREATE OR REPLACE TABLE reference AS
    SELECT id_individu, nom FROM mesure USING SAMPLE 50 PERCENT (bernoulli, 1)
""")
lignes_completes = con.sql("SELECT COUNT(*) FROM mesure").fetchone()[0]
print(f"{lignes_completes:,} lignes".replace(",", " "))

mesures_completes = []

mesures_completes.append(mesurer("lecture",    lambda: ...))
mesures_completes.append(mesurer("filtre",     lambda: ...))
mesures_completes.append(mesurer("agregation", lambda: ...))
mesures_completes.append(mesurer("tri",        lambda: ...))
mesures_completes.append(mesurer("jointure",   lambda: ...))

enregistrer(mesures_completes, outil="duckdb", volume="complet",
            lignes=lignes_completes)

## 9. Ce qu'il faut retenir

- DuckDB est une **base analytique sans serveur** : une bibliothèque, un
  fichier, rien à administrer. C'est un argument de poids en administration.
- Il interroge **directement** CSV et Parquet, sans étape de chargement — mais
  chaque requête relit le fichier ; au-delà de quelques requêtes, il vaut mieux
  charger une fois.
- `SUMMARIZE` remplace dix lignes de code de profilage.
- `EXPLAIN ANALYZE` montre le plan retenu et le temps de chaque étape.
- Il échange avec pandas et Polars **sans recopie** : on compose les outils au
  lieu d'en choisir un.
- Il déborde sur le disque quand la mémoire manque, dans une limite que l'on
  fixe.

**À compléter :**

- Requête sur CSV : … s · sur table chargée : … s · rentabilité à partir de … requêtes
- Opération la plus rapide : … · la plus lente : …
- Échecs sur le fichier complet : …
